# US Accidents — Data Preprocessing Pipeline (Converted from src/*.py)
---
**File:** `03_preprocessing.ipynb`
**Purpose:** Full Stage 4 pipeline, converted from teammate's script files, in the correct split-first order.
**Input:** `../data/US_Accidents_1M.csv`
**Key principle:** Train/test split happens FIRST. Every statistic used for imputation, encoding, or outlier treatment (medians, frequency maps, invalid-value bounds) is computed from the TRAINING SET ONLY, then applied to both train and test. This avoids leakage from test-set statistics into preprocessing decisions.

**Pipeline order:**
1. Train/Test Split (stratified)
2. Duplicate Record Analysis
3. Missing Value Analysis (diagnostic)
4. Missing Value Treatment Plan (decisions)
5. Apply Missing Value Treatment
6. Invalid Value Analysis (diagnostic)
7. Invalid Value Treatment
8. Temporal Feature Engineering
9. Categorical Feature Analysis (diagnostic)
10. Apply Categorical Encoding
11. Numerical Feature Analysis (diagnostic)
12. Apply Numerical Transformation
13. Final Feature Selection
---

## 0. Feature Audit (Stage 4.1)
Documents an initial decision and reason for every column in the raw sampled dataset, based on Stage 3 EDA findings. This runs BEFORE the train/test split since it's auditing the dataset's columns, not computing any statistics.

In [1]:
import pandas as pd
from pathlib import Path

INPUT_FILE = Path("../data/US_Accidents_1M.csv")
OUTPUT_FILE = Path("../outputs/reports/feature_audit.csv")

print("Loading dataset...")
df = pd.read_csv(INPUT_FILE)
print(f"Dataset shape: {df.shape}")

TARGET = "Severity"

decisions = {
    "Severity": ("TARGET", "Target variable for multiclass classification"),
    "ID": ("EXCLUDE", "Identifier with no meaningful predictive information"),
    "Country": ("EXCLUDE", "Constant column; contains no variation"),
    "Turning_Loop": ("EXCLUDE", "Constant column; contains no variation"),
    "End_Time": ("LEAKAGE_RISK", "Potential post-event information unavailable at prediction time"),
    "End_Lat": ("LEAKAGE_RISK", "Potential post-event information; substantially missing (see Missing_Percentage)"),
    "End_Lng": ("LEAKAGE_RISK", "Potential post-event information; substantially missing (see Missing_Percentage)"),
    "Source": ("INVESTIGATE", "Record source with few categories; may reflect data-collection differences or source bias"),
    "Start_Time": ("TEMPORAL", "Used to derive hour, day, month, year, weekday, weekend, night and cyclical time features"),
    "Weather_Timestamp": ("INVESTIGATE", "Weather observation time must be checked for prediction-time availability"),
    "Start_Lat": ("GEOGRAPHIC", "Accident starting latitude; potentially useful for spatial patterns"),
    "Start_Lng": ("GEOGRAPHIC", "Accident starting longitude; potentially useful for spatial patterns"),
    "Street": ("HIGH_CARDINALITY", "Very many unique values (see Unique_Values); investigate frequency encoding or alternative representation"),
    "City": ("CATEGORICAL", "Categorical geographic variable; encoding strategy to be determined"),
    "County": ("CATEGORICAL", "Categorical geographic variable; encoding strategy to be determined"),
    "State": ("CATEGORICAL", "Low-cardinality geographic variable suitable for categorical encoding"),
    "Zipcode": ("HIGH_CARDINALITY", "Very many unique values (see Unique_Values); investigate frequency encoding or exclusion"),
    "Timezone": ("CATEGORICAL", "Low-cardinality categorical variable"),
    "Airport_Code": ("HIGH_CARDINALITY", "High-cardinality (see Unique_Values); investigate usefulness and encoding"),
    "Description": ("TEXT_INVESTIGATE", "High-cardinality free-text field; possible post-event information; NLP not included in initial pipeline"),
    "Distance(mi)": ("NUMERICAL_INVESTIGATE", "Strong right skew; investigate log1p transformation and prediction-time availability"),
    "Temperature(F)": ("WEATHER_NUMERICAL", "Weather variable with low missingness"),
    "Wind_Chill(F)": ("WEATHER_NUMERICAL", "High missingness; investigate missingness indicator and imputation"),
    "Humidity(%)": ("WEATHER_NUMERICAL", "Weather variable with low missingness"),
    "Pressure(in)": ("WEATHER_NUMERICAL", "Contains suspicious extreme values; requires domain-aware validation"),
    "Visibility(mi)": ("WEATHER_NUMERICAL", "Highly concentrated at 10 miles; IQR method is unsuitable for direct outlier removal"),
    "Wind_Direction": ("WEATHER_CATEGORICAL", "Categorical weather variable with low missingness"),
    "Wind_Speed(mph)": ("WEATHER_NUMERICAL", "Contains suspicious extreme value of 822.8 mph; requires validation"),
    "Precipitation(in)": ("WEATHER_NUMERICAL", "High missingness and strongly zero-inflated; investigate missingness indicator and log1p"),
    "Weather_Condition": ("WEATHER_CATEGORICAL", "Categorical weather variable with low missingness"),
    "Amenity": ("BOOLEAN", "Road/environment context indicator"),
    "Bump": ("BOOLEAN", "Road/environment context indicator"),
    "Crossing": ("BOOLEAN", "Road/environment context indicator"),
    "Give_Way": ("BOOLEAN", "Road/environment context indicator"),
    "Junction": ("BOOLEAN", "Road/environment context indicator"),
    "No_Exit": ("BOOLEAN", "Road/environment context indicator"),
    "Railway": ("BOOLEAN", "Road/environment context indicator"),
    "Roundabout": ("BOOLEAN", "Road/environment context indicator"),
    "Station": ("BOOLEAN", "Road/environment context indicator"),
    "Stop": ("BOOLEAN", "Road/environment context indicator"),
    "Traffic_Calming": ("BOOLEAN", "Road/environment context indicator"),
    "Traffic_Signal": ("BOOLEAN", "Road/environment context indicator"),
    "Sunrise_Sunset": ("TEMPORAL_CATEGORICAL", "Day/night context; potentially redundant with Start_Time-derived features"),
    "Civil_Twilight": ("TEMPORAL_CATEGORICAL", "Twilight context; investigate redundancy with derived temporal features"),
    "Nautical_Twilight": ("TEMPORAL_CATEGORICAL", "Twilight context; investigate redundancy with derived temporal features"),
    "Astronomical_Twilight": ("TEMPORAL_CATEGORICAL", "Twilight context; investigate redundancy with derived temporal features"),
}

dataset_columns = set(df.columns)
audited_columns = set(decisions.keys())
missing_from_audit = dataset_columns - audited_columns
extra_in_audit = audited_columns - dataset_columns

if missing_from_audit:
    raise ValueError(f"Columns missing from feature audit: {sorted(missing_from_audit)}")
if extra_in_audit:
    raise ValueError(f"Columns in audit but not dataset: {sorted(extra_in_audit)}")

rows = []
for column in df.columns:
    decision, reason = decisions[column]
    rows.append({
        "Feature": column, "Data_Type": str(df[column].dtype),
        "Unique_Values": df[column].nunique(dropna=True),
        "Missing_Count": int(df[column].isna().sum()),
        "Missing_Percentage": round(df[column].isna().mean() * 100, 2),
        "Decision": decision, "Reason": reason
    })

audit_df = pd.DataFrame(rows)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
audit_df.to_csv(OUTPUT_FILE, index=False)

print("Feature audit completed.")
print(audit_df.to_string(index=False))
print("\nDecision summary:")
print(audit_df["Decision"].value_counts())
print(f"\nSaved report to: {OUTPUT_FILE}")

Loading dataset...
Dataset shape: (999592, 46)
Feature audit completed.
              Feature Data_Type  Unique_Values  Missing_Count  Missing_Percentage              Decision                                                                                                    Reason
                   ID    object         999592              0                0.00               EXCLUDE                                                      Identifier with no meaningful predictive information
               Source    object              3              0                0.00           INVESTIGATE                 Record source with few categories; may reflect data-collection differences or source bias
             Severity     int64              4              0                0.00                TARGET                                                             Target variable for multiclass classification
           Start_Time    object         951934              0                0.00       

## 1. Train/Test Split (Stratified)
Splits BEFORE any cleaning/imputation, so no preprocessing statistic is ever computed on data that includes the test set.

In [2]:
import os
from sklearn.model_selection import train_test_split

INPUT_FILE = "../data/US_Accidents_1M.csv"
TRAIN_FILE = "../data/train.csv"
TEST_FILE = "../data/test.csv"
REPORT_FILE = "../outputs/reports/train_test_distribution.csv"
TEST_SIZE = 0.20
RANDOM_SEED = 42

print("Loading dataset...")
df = pd.read_csv(INPUT_FILE)
print(f"Dataset shape: {df.shape}")

TARGET = "Severity"
X = df.drop(columns=[TARGET])
y = df[TARGET]

print("\nCreating stratified train/test split...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED
)

train_df = X_train.copy()
train_df[TARGET] = y_train
test_df = X_test.copy()
test_df[TARGET] = y_test

os.makedirs("../data", exist_ok=True)
os.makedirs("../outputs/reports", exist_ok=True)

train_df.to_csv(TRAIN_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)

full_distribution = (y.value_counts(normalize=True).sort_index() * 100)
train_distribution = (y_train.value_counts(normalize=True).sort_index() * 100)
test_distribution = (y_test.value_counts(normalize=True).sort_index() * 100)

distribution_report = pd.DataFrame({
    "Full_Percentage": full_distribution,
    "Train_Percentage": train_distribution,
    "Test_Percentage": test_distribution,
})
distribution_report.index.name = "Severity"
distribution_report["Full_Count"] = y.value_counts().sort_index()
distribution_report["Train_Count"] = y_train.value_counts().sort_index()
distribution_report["Test_Count"] = y_test.value_counts().sort_index()
distribution_report = distribution_report[
    ["Full_Count", "Train_Count", "Test_Count", "Full_Percentage", "Train_Percentage", "Test_Percentage"]
]
distribution_report.to_csv(REPORT_FILE)

print("\nTrain/Test split completed.")
print(f"\nTraining set shape: {train_df.shape}")
print(f"Testing set shape:  {test_df.shape}")
print("\nSeverity distribution:")
print(distribution_report.to_string(float_format=lambda x: f"{x:.4f}"))

Loading dataset...
Dataset shape: (999592, 46)

Creating stratified train/test split...

Train/Test split completed.

Training set shape: (799673, 46)
Testing set shape:  (199919, 46)

Severity distribution:
          Full_Count  Train_Count  Test_Count  Full_Percentage  Train_Percentage  Test_Percentage
Severity                                                                                         
1               8612         6890        1722           0.8616            0.8616           0.8613
2             796136       636908      159228          79.6461           79.6461          79.6463
3             168682       134945       33737          16.8751           16.8750          16.8753
4              26162        20930        5232           2.6173            2.6173           2.6171


## 2. Duplicate Record Analysis
Checks for exact duplicate rows, duplicate IDs, and train/test ID overlap (a leakage check — no record should appear in both sets).

In [3]:
TRAIN_FILE = "../data/train.csv"
TEST_FILE = "../data/test.csv"
OUTPUT_FILE = "../outputs/reports/duplicate_analysis.csv"

os.makedirs("../outputs/reports", exist_ok=True)

print("Loading training dataset...")
train = pd.read_csv(TRAIN_FILE)
print(f"Training dataset shape: {train.shape}")

print("\nLoading testing dataset...")
test = pd.read_csv(TEST_FILE)
print(f"Testing dataset shape: {test.shape}")

train_duplicates = int(train.duplicated().sum())
test_duplicates = int(test.duplicated().sum())
print(f"\nTraining exact duplicate rows: {train_duplicates:,}")
print(f"Testing exact duplicate rows : {test_duplicates:,}")

train_duplicate_ids = int(train["ID"].duplicated().sum())
test_duplicate_ids = int(test["ID"].duplicated().sum())
print(f"\nTraining duplicate IDs: {train_duplicate_ids:,}")
print(f"Testing duplicate IDs : {test_duplicate_ids:,}")

train_ids = set(train["ID"])
test_ids = set(test["ID"])
overlap_count = len(train_ids.intersection(test_ids))
print(f"\nIDs appearing in both training and testing sets: {overlap_count:,}")

report = pd.DataFrame({
    "Check": ["Training exact duplicate rows", "Testing exact duplicate rows",
              "Training duplicate IDs", "Testing duplicate IDs", "Train/Test overlapping IDs"],
    "Count": [train_duplicates, test_duplicates, train_duplicate_ids, test_duplicate_ids, overlap_count],
    "Treatment": [
        "No removal required" if train_duplicates == 0 else "Remove duplicates",
        "No removal required" if test_duplicates == 0 else "Remove duplicates",
        "No removal required" if train_duplicate_ids == 0 else "Investigate duplicate IDs",
        "No removal required" if test_duplicate_ids == 0 else "Investigate duplicate IDs",
        "No leakage detected" if overlap_count == 0 else "Investigate train/test overlap"
    ]
})
report.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved report: {OUTPUT_FILE}")

Loading training dataset...
Training dataset shape: (799673, 46)

Loading testing dataset...
Testing dataset shape: (199919, 46)

Training exact duplicate rows: 0
Testing exact duplicate rows : 0

Training duplicate IDs: 0
Testing duplicate IDs : 0

IDs appearing in both training and testing sets: 0

Saved report: ../outputs/reports/duplicate_analysis.csv


## 3. Missing Value Analysis (Diagnostic)
Profiles missingness on the TRAINING set only — classifies each column's missingness severity (VERY_LOW to VERY_HIGH).

In [4]:
INPUT_FILE = "../data/train.csv"
REPORT_FILE = "../outputs/reports/train_missing_value_analysis.csv"

print("Loading training dataset...")
df = pd.read_csv(INPUT_FILE)
print(f"Training dataset shape: {df.shape}")

missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

analysis = pd.DataFrame({
    "Feature": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Count": missing_count.values,
    "Missing_Percentage": missing_percentage.values,
    "Unique_Values": [df[col].nunique(dropna=True) for col in df.columns]
})

analysis = analysis[analysis["Missing_Count"] > 0].copy()
analysis = analysis.sort_values(by="Missing_Percentage", ascending=False)

def classify_missingness(percentage):
    if percentage == 0:
        return "NONE"
    elif percentage < 1:
        return "VERY_LOW"
    elif percentage < 5:
        return "LOW"
    elif percentage < 20:
        return "MODERATE"
    elif percentage < 50:
        return "HIGH"
    else:
        return "VERY_HIGH"

analysis["Missingness_Level"] = analysis["Missing_Percentage"].apply(classify_missingness)

os.makedirs("../outputs/reports", exist_ok=True)
analysis.to_csv(REPORT_FILE, index=False)

print("\nTraining-set missing value analysis:")
print(analysis.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nMissingness level summary:")
print(analysis["Missingness_Level"].value_counts().to_string())

Loading training dataset...
Training dataset shape: (799673, 46)

Training-set missing value analysis:
              Feature Data_Type  Missing_Count  Missing_Percentage  Unique_Values Missingness_Level
              End_Lat   float64         352033             44.0221         325899              HIGH
              End_Lng   float64         352033             44.0221         328102              HIGH
    Precipitation(in)   float64         228510             28.5754            194              HIGH
        Wind_Chill(F)   float64         207445             25.9412            846              HIGH
      Wind_Speed(mph)   float64          59424              7.4310            110          MODERATE
       Visibility(mi)   float64          18525              2.3166             61               LOW
       Wind_Direction    object          18260              2.2834             24               LOW
    Weather_Condition    object          18160              2.2709            113               L

## 4. Missing Value Treatment Plan (Decisions)
Documents the per-column decision (exclude / impute with indicator / median / missing-category) BEFORE applying it, with justification for each.

In [5]:
INPUT_FILE = "../data/train.csv"
OUTPUT_FILE = "../outputs/reports/missing_value_treatment_plan.csv"

print("Loading training dataset...")
df = pd.read_csv(INPUT_FILE)
print(f"Training dataset shape: {df.shape}")

decisions = {
    "End_Lat": ("EXCLUDE", "Potential post-event information and substantially missing; exclude rather than impute to avoid leakage."),
    "End_Lng": ("EXCLUDE", "Potential post-event information and substantially missing; exclude rather than impute to avoid leakage."),
    "Precipitation(in)": ("MISSING_INDICATOR_AND_MEDIAN", "High missingness and zero-inflated distribution; retain missingness information using an indicator and impute using training-set median."),
    "Wind_Chill(F)": ("MISSING_INDICATOR_AND_MEDIAN", "High missingness; retain missingness information and use training-set median imputation."),
    "Wind_Speed(mph)": ("MISSING_INDICATOR_AND_MEDIAN", "Moderate missingness and suspicious extreme values; retain missingness information and use training-set median imputation after invalid-value investigation."),
    "Visibility(mi)": ("MEDIAN", "Low missingness; median imputation is robust to the concentrated distribution around 10 miles."),
    "Humidity(%)": ("MEDIAN", "Low missingness; median imputation is appropriate for a bounded numerical weather variable."),
    "Temperature(F)": ("MEDIAN", "Low missingness; median imputation is appropriate for the numerical weather variable."),
    "Pressure(in)": ("MEDIAN", "Low missingness; median imputation is robust to the suspicious extreme values identified during EDA."),
    "Weather_Condition": ("MISSING_CATEGORY", "Categorical weather variable; represent missing observations explicitly rather than assuming the most common weather condition."),
    "Wind_Direction": ("MISSING_CATEGORY", "Categorical weather variable; represent missing observations as a separate category."),
    "Weather_Timestamp": ("INVESTIGATE", "Missingness is low, but prediction-time availability must be verified before deciding whether to retain the feature."),
    "Airport_Code": ("MISSING_CATEGORY", "Low missingness; preserve missingness explicitly while high-cardinality usefulness is evaluated."),
    "Street": ("MISSING_CATEGORY", "Very low missingness but very high cardinality; represent missing values explicitly before deciding the final encoding strategy."),
    "Timezone": ("MISSING_CATEGORY", "Very low missingness and only four categories; represent missing observations explicitly."),
    "Zipcode": ("MISSING_CATEGORY", "Very low missingness but high cardinality; represent missing observations explicitly before encoding."),
    "City": ("MISSING_CATEGORY", "Extremely low missingness; use an explicit missing category for consistency."),
    "Sunrise_Sunset": ("MISSING_CATEGORY", "Very low missingness; represent missing values explicitly and later assess redundancy with Start_Time-derived day/night features."),
    "Civil_Twilight": ("MISSING_CATEGORY", "Very low missingness; represent missing values explicitly and later assess redundancy."),
    "Nautical_Twilight": ("MISSING_CATEGORY", "Very low missingness; represent missing values explicitly and later assess redundancy."),
    "Astronomical_Twilight": ("MISSING_CATEGORY", "Very low missingness; represent missing values explicitly and later assess redundancy."),
    "Description": ("MISSING_CATEGORY", "Very low missingness; if the text feature is retained for a later NLP experiment, missing text can be represented explicitly."),
}

missing_columns = set(df.columns[df.isnull().any()])
decision_columns = set(decisions.keys())
missing_decisions = missing_columns - decision_columns

if missing_decisions:
    print(f"WARNING - no treatment decision defined for: {sorted(missing_decisions)}")

rows = []
for feature, (decision, reason) in decisions.items():
    if feature not in df.columns:
        continue
    missing_count = df[feature].isna().sum()
    missing_percentage = (missing_count / len(df)) * 100
    rows.append({
        "Feature": feature, "Missing_Count": missing_count,
        "Missing_Percentage": missing_percentage, "Treatment": decision, "Reason": reason
    })

treatment_plan = pd.DataFrame(rows).sort_values(by="Missing_Percentage", ascending=False)
os.makedirs("../outputs/reports", exist_ok=True)
treatment_plan.to_csv(OUTPUT_FILE, index=False)

print("\nMissing-value treatment plan:")
print(treatment_plan.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nTreatment summary:")
print(treatment_plan["Treatment"].value_counts().to_string())

Loading training dataset...
Training dataset shape: (799673, 46)

Missing-value treatment plan:
              Feature  Missing_Count  Missing_Percentage                    Treatment                                                                                                                                                       Reason
              End_Lat         352033             44.0221                      EXCLUDE                                                     Potential post-event information and substantially missing; exclude rather than impute to avoid leakage.
              End_Lng         352033             44.0221                      EXCLUDE                                                     Potential post-event information and substantially missing; exclude rather than impute to avoid leakage.
    Precipitation(in)         228510             28.5754 MISSING_INDICATOR_AND_MEDIAN                     High missingness and zero-inflated distribution; retain missingness i

## 5. Apply Missing Value Treatment
Executes the plan above: drops leakage columns, creates missingness indicators, imputes numeric columns with TRAINING medians (applied to both sets), and fills categorical gaps with an explicit "Missing" category.

In [6]:
TRAIN_INPUT = "../data/train.csv"
TEST_INPUT = "../data/test.csv"
TRAIN_OUTPUT = "../data/train_missing_handled.csv"
TEST_OUTPUT = "../data/test_missing_handled.csv"
REPORT_OUTPUT = "../outputs/reports/missing_value_treatment_results.csv"

LEAKAGE_FEATURES = ["End_Lat", "End_Lng"]
UNRESOLVED_FEATURES = ["Weather_Timestamp", "Description"]
NUMERIC_INDICATOR_FEATURES = ["Precipitation(in)", "Wind_Chill(F)", "Wind_Speed(mph)"]
NUMERIC_MEDIAN_FEATURES = ["Precipitation(in)", "Wind_Chill(F)", "Wind_Speed(mph)",
                            "Visibility(mi)", "Humidity(%)", "Temperature(F)", "Pressure(in)"]
CATEGORICAL_FEATURES = ["Weather_Condition", "Wind_Direction", "Airport_Code", "Street",
                         "Timezone", "Zipcode", "City", "Sunrise_Sunset", "Civil_Twilight",
                         "Nautical_Twilight", "Astronomical_Twilight"]

print("Loading training dataset...")
train_df = pd.read_csv(TRAIN_INPUT)
print(f"Training dataset shape: {train_df.shape}")

print("\nLoading testing dataset...")
test_df = pd.read_csv(TEST_INPUT)
print(f"Testing dataset shape: {test_df.shape}")

features_to_remove = LEAKAGE_FEATURES + UNRESOLVED_FEATURES
print("\nRemoving features from the initial pipeline:")
for feature in features_to_remove:
    print(f"- {feature}")

train_df = train_df.drop(columns=features_to_remove, errors="ignore")
test_df = test_df.drop(columns=features_to_remove, errors="ignore")

print("\nCreating missingness indicators...")
for feature in NUMERIC_INDICATOR_FEATURES:
    indicator_name = feature.replace("(", "_").replace(")", "").replace("/", "_").replace(" ", "_")
    indicator_name = f"{indicator_name}_Missing"
    train_df[indicator_name] = train_df[feature].isna().astype("int8")
    test_df[indicator_name] = test_df[feature].isna().astype("int8")
    print(f"- Created {indicator_name}")

print("\nCalculating training-set medians...")
training_medians = {}
for feature in NUMERIC_MEDIAN_FEATURES:
    median_value = train_df[feature].median()
    training_medians[feature] = median_value
    print(f"- {feature}: {median_value:.6f}")

print("\nApplying numerical median imputation...")
for feature, median_value in training_medians.items():
    train_df[feature] = train_df[feature].fillna(median_value)
    test_df[feature] = test_df[feature].fillna(median_value)
    print(f"- Imputed {feature}")

print("\nApplying categorical missing-category treatment...")
for feature in CATEGORICAL_FEATURES:
    train_df[feature] = train_df[feature].fillna("Missing")
    test_df[feature] = test_df[feature].fillna("Missing")
    print(f"- Processed {feature}")

print("\nChecking remaining missing values...")
train_missing = train_df.isna().sum()
test_missing = test_df.isna().sum()
remaining_train_missing = train_missing[train_missing > 0].sort_values(ascending=False)
remaining_test_missing = test_missing[test_missing > 0].sort_values(ascending=False)

print("\nRemaining missing values in training data:")
print("None" if remaining_train_missing.empty else remaining_train_missing.to_string())
print("\nRemaining missing values in testing data:")
print("None" if remaining_test_missing.empty else remaining_test_missing.to_string())

report_rows = []
for feature in train_df.columns:
    train_missing_count = train_df[feature].isna().sum()
    test_missing_count = test_df[feature].isna().sum()
    report_rows.append({
        "Feature": feature, "Train_Missing_Count": train_missing_count, "Test_Missing_Count": test_missing_count,
        "Train_Missing_Percentage": (train_missing_count / len(train_df)) * 100,
        "Test_Missing_Percentage": (test_missing_count / len(test_df)) * 100,
    })
treatment_results = pd.DataFrame(report_rows)

os.makedirs("../data", exist_ok=True)
os.makedirs("../outputs/reports", exist_ok=True)

train_df.to_csv(TRAIN_OUTPUT, index=False)
test_df.to_csv(TEST_OUTPUT, index=False)
treatment_results.to_csv(REPORT_OUTPUT, index=False)

print("\nMissing-value treatment completed.")
print(f"\nProcessed training shape: {train_df.shape}")
print(f"Processed testing shape:  {test_df.shape}")

Loading training dataset...
Training dataset shape: (799673, 46)

Loading testing dataset...
Testing dataset shape: (199919, 46)

Removing features from the initial pipeline:
- End_Lat
- End_Lng
- Weather_Timestamp
- Description

Creating missingness indicators...
- Created Precipitation_in_Missing
- Created Wind_Chill_F_Missing
- Created Wind_Speed_mph_Missing

Calculating training-set medians...
- Precipitation(in): 0.000000
- Wind_Chill(F): 62.000000
- Wind_Speed(mph): 7.000000
- Visibility(mi): 10.000000
- Humidity(%): 67.000000
- Temperature(F): 64.000000
- Pressure(in): 29.860000

Applying numerical median imputation...
- Imputed Precipitation(in)
- Imputed Wind_Chill(F)
- Imputed Wind_Speed(mph)
- Imputed Visibility(mi)
- Imputed Humidity(%)
- Imputed Temperature(F)
- Imputed Pressure(in)

Applying categorical missing-category treatment...
- Processed Weather_Condition
- Processed Wind_Direction
- Processed Airport_Code
- Processed Street
- Processed Timezone
- Processed Zipcode

## 6. Invalid Value Analysis (Diagnostic)
Investigates physically implausible values (extreme temperature, negative precipitation, etc.) on the TRAINING set — reports counts and cross-tabs by severity/year/state without modifying data yet.

In [7]:
from pathlib import Path

INPUT_FILE = Path("../data/train_missing_handled.csv")
OUTPUT_DIR = Path("../outputs/reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
print(f"\nTraining dataset shape: {df.shape}")

checks = {
    "Temperature(F)": {"condition": (df["Temperature(F)"] < -80) | (df["Temperature(F)"] > 140), "reason": "Physically unusual temperature"},
    "Wind_Speed(mph)": {"condition": (df["Wind_Speed(mph)"] < 0) | (df["Wind_Speed(mph)"] > 100), "reason": "Physically questionable wind speed"},
    "Pressure(in)": {"condition": (df["Pressure(in)"] < 20) | (df["Pressure(in)"] > 35), "reason": "Physically questionable atmospheric pressure"},
    "Visibility(mi)": {"condition": (df["Visibility(mi)"] < 0) | (df["Visibility(mi)"] > 100), "reason": "Outside expected visibility range"},
    "Humidity(%)": {"condition": (df["Humidity(%)"] < 0) | (df["Humidity(%)"] > 100), "reason": "Outside valid percentage range"},
    "Precipitation(in)": {"condition": df["Precipitation(in)"] < 0, "reason": "Negative precipitation"},
    "Distance(mi)": {"condition": (df["Distance(mi)"] < 0) | (df["Distance(mi)"] > 200), "reason": "Negative or extremely large distance"}
}

results = []
print("\nSuspicious-value summary:")
for feature, info in checks.items():
    condition = info["condition"].fillna(False)
    count = int(condition.sum())
    percentage = (count / len(df)) * 100
    results.append({"Feature": feature, "Suspicious_Count": count, "Suspicious_Percentage": round(percentage, 4), "Reason": info["reason"]})
    print(f"{feature:<22} {count:>8,} records ({percentage:>7.4f}%)")

results_df = pd.DataFrame(results)
summary_file = OUTPUT_DIR / "invalid_value_summary.csv"
results_df.to_csv(summary_file, index=False)
print(f"\nSaved: {summary_file}")

combined_suspicious = pd.Series(False, index=df.index)
for feature, info in checks.items():
    combined_suspicious = combined_suspicious | info["condition"].fillna(False)

combined_count = int(combined_suspicious.sum())
combined_percentage = (combined_count / len(df)) * 100
print(f"\nRecords with at least one suspicious value: {combined_count:,} ({combined_percentage:.4f}%)")

suspicious_columns = ["ID", "Severity", "Start_Time", "State", "Source", "Temperature(F)",
                      "Wind_Speed(mph)", "Pressure(in)", "Visibility(mi)", "Precipitation(in)", "Distance(mi)"]
suspicious_columns = [c for c in suspicious_columns if c in df.columns]
suspicious_records = df.loc[combined_suspicious, suspicious_columns]
suspicious_file = OUTPUT_DIR / "suspicious_records.csv"
suspicious_records.to_csv(suspicious_file, index=False)
print(f"\nSaved: {suspicious_file}")


Training dataset shape: (799673, 45)

Suspicious-value summary:
Temperature(F)                3 records ( 0.0004%)
Wind_Speed(mph)              15 records ( 0.0019%)
Pressure(in)                 44 records ( 0.0055%)
Visibility(mi)                1 records ( 0.0001%)
Humidity(%)                   0 records ( 0.0000%)
Precipitation(in)             0 records ( 0.0000%)
Distance(mi)                  0 records ( 0.0000%)

Saved: ../outputs/reports/invalid_value_summary.csv

Records with at least one suspicious value: 63 (0.0079%)

Saved: ../outputs/reports/suspicious_records.csv


## 7. Invalid Value Treatment
Replaces out-of-range values with NaN, then imputes using TRAINING-set medians only (applied to both train and test).

In [8]:
TRAIN_FILE = Path("../data/train_missing_handled.csv")
TEST_FILE = Path("../data/test_missing_handled.csv")
OUTPUT_DIR = Path("../outputs/reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_OUTPUT = Path("../data/train_outliers_handled.csv")
TEST_OUTPUT = Path("../data/test_outliers_handled.csv")

train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE)
print(f"\nTraining shape: {train.shape}")
print(f"Testing shape : {test.shape}")

invalid_rules = {"Temperature(F)": (-80, 140), "Wind_Speed(mph)": (0, 100), "Pressure(in)": (20, 35), "Visibility(mi)": (0, 100)}

replacement_values = {}
for feature in invalid_rules:
    median_value = train[feature].median()
    replacement_values[feature] = median_value
    print(f"\n{feature}\nTraining median: {median_value}")

treatment_results = []
for feature, (lower, upper) in invalid_rules.items():
    train_invalid = (train[feature] < lower) | (train[feature] > upper)
    test_invalid = (test[feature] < lower) | (test[feature] > upper)
    train_count = int(train_invalid.sum())
    test_count = int(test_invalid.sum())

    train.loc[train_invalid, feature] = pd.NA
    test.loc[test_invalid, feature] = pd.NA

    median_value = replacement_values[feature]
    train[feature] = train[feature].fillna(median_value)
    test[feature] = test[feature].fillna(median_value)

    treatment_results.append({
        "Feature": feature, "Lower_Bound": lower, "Upper_Bound": upper,
        "Training_Invalid_Count": train_count, "Testing_Invalid_Count": test_count,
        "Training_Median_Used": median_value, "Treatment": "Replace invalid values with training median"
    })

print("\nVerification:")
for feature, (lower, upper) in invalid_rules.items():
    train_remaining = ((train[feature] < lower) | (train[feature] > upper)).sum()
    test_remaining = ((test[feature] < lower) | (test[feature] > upper)).sum()
    print(f"{feature:<22} Train remaining: {train_remaining} | Test remaining: {test_remaining}")

train.to_csv(TRAIN_OUTPUT, index=False)
test.to_csv(TEST_OUTPUT, index=False)

results_df = pd.DataFrame(treatment_results)
report_file = OUTPUT_DIR / "invalid_value_treatment_results.csv"
results_df.to_csv(report_file, index=False)
print(f"\nSaved training data: {TRAIN_OUTPUT}")
print(f"Saved testing data : {TEST_OUTPUT}")
print(f"Saved treatment report: {report_file}")


Training shape: (799673, 45)
Testing shape : (199919, 45)

Temperature(F)
Training median: 64.0

Wind_Speed(mph)
Training median: 7.0

Pressure(in)
Training median: 29.86

Visibility(mi)
Training median: 10.0

Verification:
Temperature(F)         Train remaining: 0 | Test remaining: 0
Wind_Speed(mph)        Train remaining: 0 | Test remaining: 0
Pressure(in)           Train remaining: 0 | Test remaining: 0
Visibility(mi)         Train remaining: 0 | Test remaining: 0

Saved training data: ../data/train_outliers_handled.csv
Saved testing data : ../data/test_outliers_handled.csv
Saved treatment report: ../outputs/reports/invalid_value_treatment_results.csv


## 8. Temporal Feature Engineering
Extracts calendar features from Start_Time, plus weekend/night indicators and cyclical (sin/cos) encodings for hour and month.

In [9]:
import numpy as np

TRAIN_FILE = "../data/train_outliers_handled.csv"
TEST_FILE = "../data/test_outliers_handled.csv"
TRAIN_OUTPUT = "../data/train_temporal_features.csv"
TEST_OUTPUT = "../data/test_temporal_features.csv"
REPORT_FILE = "../outputs/reports/temporal_feature_engineering.csv"

os.makedirs("../outputs/reports", exist_ok=True)

def create_temporal_features(df, dataset_name):
    print(f"\nProcessing {dataset_name} dataset...")
    if "Start_Time" not in df.columns:
        raise ValueError("Start_Time column is missing from the dataset.")

    start_time = pd.to_datetime(df["Start_Time"].astype(str).str.strip(), format="mixed", errors="coerce")
    failed = int(start_time.isna().sum())
    print(f"Timestamp parsing failures: {failed:,}")
    if failed > 0:
        raise ValueError(f"{failed:,} Start_Time values could not be parsed.")

    df["Start_Year"] = start_time.dt.year
    df["Start_Month"] = start_time.dt.month
    df["Start_Day"] = start_time.dt.day
    df["Start_Hour"] = start_time.dt.hour
    df["Start_DayOfWeek"] = start_time.dt.dayofweek
    df["Is_Weekend"] = (df["Start_DayOfWeek"] >= 5).astype(int)
    df["Is_Night"] = ((df["Start_Hour"] < 6) | (df["Start_Hour"] >= 20)).astype(int)
    df["Hour_Sin"] = np.sin(2 * np.pi * df["Start_Hour"] / 24)
    df["Hour_Cos"] = np.cos(2 * np.pi * df["Start_Hour"] / 24)
    df["Month_Sin"] = np.sin(2 * np.pi * (df["Start_Month"] - 1) / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * (df["Start_Month"] - 1) / 12)
    return df, start_time

print("\nLoading training dataset...")
train = pd.read_csv(TRAIN_FILE)
print(f"Training dataset shape: {train.shape}")

print("\nLoading testing dataset...")
test = pd.read_csv(TEST_FILE)
print(f"Testing dataset shape: {test.shape}")

train, train_start_time = create_temporal_features(train, "training")
test, test_start_time = create_temporal_features(test, "testing")

temporal_columns = ["Start_Year", "Start_Month", "Start_Day", "Start_Hour", "Start_DayOfWeek",
                     "Is_Weekend", "Is_Night", "Hour_Sin", "Hour_Cos", "Month_Sin", "Month_Cos"]

print("\nTraining temporal feature summary:")
print(train[temporal_columns].describe().T)

train_missing = train[temporal_columns].isna().sum().sum()
test_missing = test[temporal_columns].isna().sum().sum()
print(f"\nMissing temporal feature values: Training: {train_missing:,} | Testing: {test_missing:,}")
if train_missing > 0 or test_missing > 0:
    raise ValueError("Missing values detected in engineered temporal features.")

train.to_csv(TRAIN_OUTPUT, index=False)
test.to_csv(TEST_OUTPUT, index=False)

descriptions = {
    "Start_Year": "Year of accident start time", "Start_Month": "Month of accident start time",
    "Start_Day": "Day of month of accident start time", "Start_Hour": "Hour of accident start time",
    "Start_DayOfWeek": "Day of week; Monday=0, Sunday=6", "Is_Weekend": "Indicator for Saturday or Sunday",
    "Is_Night": "Indicator for nighttime accident", "Hour_Sin": "Cyclical sine encoding of hour",
    "Hour_Cos": "Cyclical cosine encoding of hour", "Month_Sin": "Cyclical sine encoding of month",
    "Month_Cos": "Cyclical cosine encoding of month"
}
report = pd.DataFrame([{
    "Feature": f, "Data_Type": str(train[f].dtype), "Unique_Values": int(train[f].nunique()),
    "Missing_Count": int(train[f].isna().sum()), "Training_Min": train[f].min(), "Training_Max": train[f].max(),
    "Description": descriptions[f]
} for f in temporal_columns])
report.to_csv(REPORT_FILE, index=False)

print(f"\nFinal dataset shapes: Training: {train.shape} | Testing: {test.shape}")


Loading training dataset...
Training dataset shape: (799673, 45)

Loading testing dataset...
Testing dataset shape: (199919, 45)

Processing training dataset...
Timestamp parsing failures: 0

Processing testing dataset...
Timestamp parsing failures: 0

Training temporal feature summary:
                    count         mean       std     min          25%  \
Start_Year       799673.0  2019.905364  1.915228  2016.0  2018.000000   
Start_Month      799673.0     6.703953  3.639158     1.0     3.000000   
Start_Day        799673.0    15.739122  8.683087     1.0     8.000000   
Start_Hour       799673.0    12.333658  5.469517     0.0     8.000000   
Start_DayOfWeek  799673.0     2.597142  1.806348     0.0     1.000000   
Is_Weekend       799673.0     0.159563  0.366200     0.0     0.000000   
Is_Night         799673.0     0.193008  0.394659     0.0     0.000000   
Hour_Sin         799673.0    -0.057737  0.759201    -1.0    -0.866025   
Hour_Cos         799673.0    -0.283556  0.582990    -1

## 9. Categorical Feature Analysis (Diagnostic)
Determines the encoding strategy (ONE_HOT / FREQUENCY / EXCLUDE) for each categorical column based on cardinality, before applying anything.

In [10]:
TRAIN_FILE = "../data/train_temporal_features.csv"
TEST_FILE = "../data/test_temporal_features.csv"
OUTPUT_FILE = "../outputs/reports/categorical_feature_analysis.csv"

os.makedirs("../outputs/reports", exist_ok=True)

print("\nLoading training dataset...")
train = pd.read_csv(TRAIN_FILE)
print(f"Training dataset shape: {train.shape}")
print("\nLoading testing dataset...")
test = pd.read_csv(TEST_FILE)
print(f"Testing dataset shape: {test.shape}")

categorical_features = ["Source", "Street", "City", "County", "State", "Zipcode", "Country",
                         "Timezone", "Airport_Code", "Wind_Direction", "Weather_Condition",
                         "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]

def determine_encoding(unique_count, feature):
    if unique_count <= 1:
        return "EXCLUDE", "Constant feature with no predictive variation."
    if feature == "Country":
        return "EXCLUDE", "Constant geographic feature; no useful variation."
    if unique_count <= 10:
        return "ONE_HOT", "Low-cardinality categorical feature suitable for one-hot encoding."
    if unique_count <= 50:
        return "ONE_HOT", "Moderate/low cardinality; one-hot encoding is feasible for the modeling pipeline."
    if feature in ["Street", "City", "Zipcode", "Airport_Code"]:
        return "FREQUENCY", "High-cardinality feature; frequency encoding avoids a very large one-hot feature matrix."
    return "FREQUENCY", "High-cardinality categorical feature; frequency encoding reduces dimensionality."

report_rows = []
for feature in categorical_features:
    train_series = train[feature].astype("string")
    test_series = test[feature].astype("string")
    train_unique = int(train_series.nunique(dropna=False))
    test_unique = int(test_series.nunique(dropna=False))
    train_missing = int(train[feature].isna().sum())
    test_missing = int(test[feature].isna().sum())

    train_categories = set(train_series.dropna().unique())
    test_categories = set(test_series.dropna().unique())
    unseen_categories = test_categories - train_categories
    unseen_count = len(unseen_categories)
    unseen_percentage = (unseen_count / train_unique * 100) if train_unique > 0 else 0.0

    encoding, reason = determine_encoding(train_unique, feature)

    print(f"\nFeature: {feature} | Train unique: {train_unique:,} | Encoding: {encoding}")

    report_rows.append({
        "Feature": feature, "Data_Type": str(train[feature].dtype),
        "Train_Unique_Values": train_unique, "Test_Unique_Values": test_unique,
        "Train_Missing_Count": train_missing, "Test_Missing_Count": test_missing,
        "Unseen_Test_Categories": unseen_count, "Unseen_Test_Percentage": round(unseen_percentage, 4),
        "Encoding_Strategy": encoding, "Reason": reason
    })

report = pd.DataFrame(report_rows)
report.to_csv(OUTPUT_FILE, index=False)

print("\nEncoding strategy summary:")
print(report[["Feature", "Train_Unique_Values", "Test_Unique_Values", "Unseen_Test_Categories", "Encoding_Strategy"]].to_string(index=False))
print("\nEncoding counts:")
print(report["Encoding_Strategy"].value_counts())


Loading training dataset...
Training dataset shape: (799673, 56)

Loading testing dataset...
Testing dataset shape: (199919, 56)

Feature: Source | Train unique: 3 | Encoding: ONE_HOT

Feature: Street | Train unique: 114,797 | Encoding: FREQUENCY

Feature: City | Train unique: 10,349 | Encoding: FREQUENCY

Feature: County | Train unique: 1,672 | Encoding: FREQUENCY

Feature: State | Train unique: 49 | Encoding: ONE_HOT

Feature: Zipcode | Train unique: 183,212 | Encoding: FREQUENCY

Feature: Country | Train unique: 1 | Encoding: EXCLUDE

Feature: Timezone | Train unique: 5 | Encoding: ONE_HOT

Feature: Airport_Code | Train unique: 1,943 | Encoding: FREQUENCY

Feature: Wind_Direction | Train unique: 25 | Encoding: ONE_HOT

Feature: Weather_Condition | Train unique: 114 | Encoding: FREQUENCY

Feature: Sunrise_Sunset | Train unique: 3 | Encoding: ONE_HOT

Feature: Civil_Twilight | Train unique: 3 | Encoding: ONE_HOT

Feature: Nautical_Twilight | Train unique: 3 | Encoding: ONE_HOT

Featu

## 10. Apply Categorical Encoding
Applies the strategy decided above: frequency encoding for high-cardinality features (mapping built from TRAINING data only), one-hot encoding for the rest (categories derived from TRAINING data; unseen test categories map to all-zero).

In [11]:
TRAIN_FILE = "../data/train_temporal_features.csv"
TEST_FILE = "../data/test_temporal_features.csv"
TRAIN_OUTPUT = "../data/train_categorical_encoded.csv"
TEST_OUTPUT = "../data/test_categorical_encoded.csv"
REPORT_OUTPUT = "../outputs/reports/categorical_encoding_results.csv"
TARGET = "Severity"

ONE_HOT_FEATURES = ["Source", "State", "Timezone", "Wind_Direction", "Sunrise_Sunset",
                     "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
FREQUENCY_FEATURES = ["Street", "City", "County", "Zipcode", "Airport_Code", "Weather_Condition"]
EXCLUDE_FEATURES = ["Country"]

def normalize_category(series):
    return series.astype("string").fillna("__MISSING__")

def build_frequency_mapping(series):
    normalized = normalize_category(series)
    return normalized.value_counts(normalize=True).to_dict()

def apply_frequency_encoding(series, mapping):
    normalized = normalize_category(series)
    return normalized.map(mapping).fillna(0.0)

print("\nLoading training dataset...")
train = pd.read_csv(TRAIN_FILE)
print(f"Training dataset shape: {train.shape}")
print("\nLoading testing dataset...")
test = pd.read_csv(TEST_FILE)
print(f"Testing dataset shape: {test.shape}")

encoding_results = []

print("\n--- FREQUENCY ENCODING ---")
for feature in FREQUENCY_FEATURES:
    print(f"\nProcessing: {feature}")
    mapping = build_frequency_mapping(train[feature])
    train[feature] = apply_frequency_encoding(train[feature], mapping)
    test[feature] = apply_frequency_encoding(test[feature], mapping)

    train_categories = set(normalize_category(pd.read_csv(TRAIN_FILE, usecols=[feature])[feature]))
    test_categories = set(normalize_category(pd.read_csv(TEST_FILE, usecols=[feature])[feature]))
    unseen_categories = test_categories - train_categories

    print(f"Training categories : {len(train_categories):,}")
    print(f"Testing categories  : {len(test_categories):,}")
    print(f"Unseen test categories: {len(unseen_categories):,}")

    encoding_results.append({
        "Feature": feature, "Encoding": "FREQUENCY", "Train_Unique_Values": len(train_categories),
        "Test_Unique_Values": len(test_categories), "Unseen_Test_Categories": len(unseen_categories),
        "Output_Type": "float64",
    })

print("\n--- ONE-HOT ENCODING ---")
for feature in ONE_HOT_FEATURES:
    print(f"\nProcessing: {feature}")
    train[feature] = normalize_category(train[feature])
    test[feature] = normalize_category(test[feature])

    train_categories = set(train[feature].unique())
    test_categories = set(test[feature].unique())
    unseen_categories = test_categories - train_categories

    train_dummies = pd.get_dummies(train[feature], prefix=feature, dtype=np.int8)
    test_dummies = pd.get_dummies(test[feature], prefix=feature, dtype=np.int8)
    test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

    print(f"Training categories : {len(train_categories):,}")
    print(f"Testing categories  : {len(test_categories):,}")
    print(f"Unseen test categories: {len(unseen_categories):,}")
    print(f"Created dummy columns: {train_dummies.shape[1]:,}")

    train.drop(columns=[feature], inplace=True)
    test.drop(columns=[feature], inplace=True)
    train = pd.concat([train, train_dummies], axis=1)
    test = pd.concat([test, test_dummies], axis=1)

    encoding_results.append({
        "Feature": feature, "Encoding": "ONE_HOT", "Train_Unique_Values": len(train_categories),
        "Test_Unique_Values": len(test_categories), "Unseen_Test_Categories": len(unseen_categories),
        "Output_Type": "int8",
    })

print("\n--- EXCLUDING CONSTANT FEATURES ---")
for feature in EXCLUDE_FEATURES:
    if feature in train.columns:
        train.drop(columns=[feature], inplace=True)
    if feature in test.columns:
        test.drop(columns=[feature], inplace=True)
    print(f"Excluded: {feature}")
    encoding_results.append({
        "Feature": feature, "Encoding": "EXCLUDE", "Train_Unique_Values": 1,
        "Test_Unique_Values": 1, "Unseen_Test_Categories": 0, "Output_Type": "excluded",
    })

train_columns = set(train.columns)
test_columns = set(test.columns)
missing_in_test = train_columns - test_columns
missing_in_train = test_columns - train_columns
if missing_in_test or missing_in_train:
    raise ValueError(f"Column mismatch. Missing in test: {missing_in_test} | Missing in train: {missing_in_train}")
test = test[train.columns]
print("\nTraining and testing columns match.")

train_missing = int(train.isna().sum().sum())
test_missing = int(test.isna().sum().sum())
print(f"\nTraining remaining missing values: {train_missing:,}")
print(f"Testing remaining missing values : {test_missing:,}")

os.makedirs(os.path.dirname(TRAIN_OUTPUT), exist_ok=True)
train.to_csv(TRAIN_OUTPUT, index=False)
test.to_csv(TEST_OUTPUT, index=False)

encoding_report = pd.DataFrame(encoding_results)
encoding_report.to_csv(REPORT_OUTPUT, index=False)

print(f"\nEncoded training shape: {train.shape}")
print(f"Encoded testing shape : {test.shape}")


Loading training dataset...
Training dataset shape: (799673, 56)

Loading testing dataset...
Testing dataset shape: (199919, 56)

--- FREQUENCY ENCODING ---

Processing: Street
Training categories : 114,797
Testing categories  : 50,761
Unseen test categories: 14,703

Processing: City
Training categories : 10,349
Testing categories  : 7,690
Unseen test categories: 381

Processing: County
Training categories : 1,672
Testing categories  : 1,456
Unseen test categories: 36

Processing: Zipcode
Training categories : 183,212
Testing categories  : 62,569
Unseen test categories: 33,027

Processing: Airport_Code
Training categories : 1,943
Testing categories  : 1,770
Unseen test categories: 21

Processing: Weather_Condition
Training categories : 114
Testing categories  : 97
Unseen test categories: 5

--- ONE-HOT ENCODING ---

Processing: Source
Training categories : 3
Testing categories  : 3
Unseen test categories: 0
Created dummy columns: 3

Processing: State
Training categories : 49
Testing c

## 11. Numerical Feature Analysis (Diagnostic)
Checks skewness, IQR-based outlier percentage, and correlation with Severity for continuous features (excluding frequency-encoded and one-hot columns, which aren't true continuous measurements).

In [12]:
TRAIN_PATH = "../data/train_categorical_encoded.csv"
TEST_PATH = "../data/test_categorical_encoded.csv"
OUTPUT_DIR = "../outputs/reports"
ANALYSIS_REPORT = os.path.join(OUTPUT_DIR, "numerical_feature_analysis.csv")
CORRELATION_REPORT = os.path.join(OUTPUT_DIR, "numerical_correlation_with_severity.csv")

CONTINUOUS_FEATURES = ["Start_Lat", "Start_Lng", "Distance(mi)", "Temperature(F)", "Wind_Chill(F)",
                        "Humidity(%)", "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"]

def iqr_outlier_percentage(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    if len(series) == 0:
        return 0.0
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower_bound) | (series > upper_bound)).mean() * 100

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("\nLoading training dataset...")
train = pd.read_csv(TRAIN_PATH)
print("Loading test dataset...")
test = pd.read_csv(TEST_PATH)

continuous_features = [c for c in CONTINUOUS_FEATURES if c in train.columns]

analysis_rows = []
for column in continuous_features:
    train_series = pd.to_numeric(train[column], errors="coerce")
    test_series = pd.to_numeric(test[column], errors="coerce")
    train_non_missing = train_series.dropna()
    if len(train_non_missing) == 0:
        continue
    q1, q3 = train_non_missing.quantile(0.25), train_non_missing.quantile(0.75)
    analysis_rows.append({
        "Feature": column, "Train_Missing_Percentage": train_series.isna().mean() * 100,
        "Mean": train_non_missing.mean(), "Median": train_non_missing.median(), "Std": train_non_missing.std(),
        "Min": train_non_missing.min(), "Q1": q1, "Q3": q3, "Max": train_non_missing.max(), "IQR": q3 - q1,
        "Skewness": train_non_missing.skew(), "IQR_Outlier_Percentage": iqr_outlier_percentage(train_series),
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_df["Absolute_Skewness"] = analysis_df["Skewness"].abs()
analysis_df = analysis_df.sort_values("Absolute_Skewness", ascending=False)
analysis_df.to_csv(ANALYSIS_REPORT, index=False)

print("\nContinuous features with |skewness| > 1:")
highly_skewed = analysis_df[analysis_df["Absolute_Skewness"] > 1]
for _, row in highly_skewed.iterrows():
    print(f"  {row['Feature']}: {row['Skewness']:.4f}")

severity = pd.to_numeric(train["Severity"], errors="coerce")
correlation_rows = []
for column in continuous_features:
    feature = pd.to_numeric(train[column], errors="coerce")
    valid = pd.DataFrame({"feature": feature, "severity": severity}).dropna()
    if len(valid) == 0:
        continue
    pearson = valid["feature"].corr(valid["severity"], method="pearson")
    spearman = valid["feature"].corr(valid["severity"], method="spearman")
    correlation_rows.append({"Feature": column, "Pearson_Correlation": pearson, "Spearman_Correlation": spearman})

correlation_df = pd.DataFrame(correlation_rows).sort_values("Spearman_Correlation", key=lambda x: x.abs(), ascending=False)
correlation_df.to_csv(CORRELATION_REPORT, index=False)

print("\nContinuous features ranked by absolute Spearman correlation with Severity:")
for _, row in correlation_df.iterrows():
    print(f"  {row['Feature']}: Spearman={row['Spearman_Correlation']:.4f}, Pearson={row['Pearson_Correlation']:.4f}")


Loading training dataset...
Loading test dataset...

Continuous features with |skewness| > 1:
  Precipitation(in): 131.3192
  Distance(mi): 16.6047
  Pressure(in): -3.7125
  Visibility(mi): 2.3015

Continuous features ranked by absolute Spearman correlation with Severity:
  Distance(mi): Spearman=-0.1413, Pearson=0.0353
  Pressure(in): Spearman=0.0904, Pearson=0.0447
  Start_Lat: Spearman=0.0597, Pearson=0.0696
  Wind_Speed(mph): Spearman=0.0444, Pearson=0.0361
  Wind_Chill(F): Spearman=-0.0437, Pearson=-0.0407
  Start_Lng: Spearman=0.0328, Pearson=0.0529
  Precipitation(in): Spearman=0.0252, Pearson=0.0062
  Humidity(%): Spearman=0.0158, Pearson=0.0215
  Visibility(mi): Spearman=-0.0104, Pearson=-0.0035
  Temperature(F): Spearman=-0.0100, Pearson=-0.0194


## 12. Apply Numerical Transformation
Applies log1p to right-skewed features (Distance, Precipitation) and removes the raw timestamp columns (temporal features already extracted).

In [13]:
TRAIN_INPUT = "../data/train_categorical_encoded.csv"
TEST_INPUT = "../data/test_categorical_encoded.csv"
TRAIN_OUTPUT = "../data/train_numerical_features.csv"
TEST_OUTPUT = "../data/test_numerical_features.csv"
REPORT_PATH = "../outputs/reports/numerical_transformation_results.csv"

os.makedirs("../outputs/reports", exist_ok=True)

print("\nLoading training dataset...")
train = pd.read_csv(TRAIN_INPUT)
print("Loading test dataset...")
test = pd.read_csv(TEST_INPUT)
print(f"\nTraining shape before transformation: {train.shape}")
print(f"Test shape before transformation:     {test.shape}")

train_distance = pd.to_numeric(train["Distance(mi)"], errors="coerce")
test_distance = pd.to_numeric(test["Distance(mi)"], errors="coerce")
train["Distance_log1p"] = np.log1p(train_distance)
test["Distance_log1p"] = np.log1p(test_distance)

train_precipitation = pd.to_numeric(train["Precipitation(in)"], errors="coerce")
test_precipitation = pd.to_numeric(test["Precipitation(in)"], errors="coerce")
train["Precipitation_log1p"] = np.log1p(train_precipitation)
test["Precipitation_log1p"] = np.log1p(test_precipitation)

print("\nRemoving raw timestamp features: Start_Time, End_Time")
train = train.drop(columns=["Start_Time", "End_Time"], errors="ignore")
test = test.drop(columns=["Start_Time", "End_Time"], errors="ignore")

transformed_features = ["Distance_log1p", "Precipitation_log1p"]
validation_rows = []
for feature in transformed_features:
    train_values = pd.to_numeric(train[feature], errors="coerce")
    test_values = pd.to_numeric(test[feature], errors="coerce")
    validation_rows.append({
        "Feature": feature, "Train_Missing": int(train_values.isna().sum()), "Test_Missing": int(test_values.isna().sum()),
        "Train_Infinite": int(np.isinf(train_values).sum()), "Test_Infinite": int(np.isinf(test_values).sum()),
        "Train_Min": train_values.min(), "Train_Max": train_values.max(),
    })
validation_df = pd.DataFrame(validation_rows)

train_missing_total = int(train.isna().sum().sum())
test_missing_total = int(test.isna().sum().sum())
print(f"\nTraining missing values: {train_missing_total}")
print(f"Test missing values:     {test_missing_total}")
if train_missing_total > 0 or test_missing_total > 0:
    raise ValueError("Missing values remain after transformation.")

if list(train.columns) != list(test.columns):
    raise ValueError("Train/test columns do not match.")
print("Train/test columns match.")

train.to_csv(TRAIN_OUTPUT, index=False)
test.to_csv(TEST_OUTPUT, index=False)
validation_df.to_csv(REPORT_PATH, index=False)

print(f"\nTraining shape after transformation: {train.shape}")
print(f"Test shape after transformation:     {test.shape}")


Loading training dataset...
Loading test dataset...

Training shape before transformation: (799673, 141)
Test shape before transformation:     (199919, 141)

Removing raw timestamp features: Start_Time, End_Time

Training missing values: 0
Test missing values:     0
Train/test columns match.

Training shape after transformation: (799673, 141)
Test shape after transformation:     (199919, 141)


## 13. Final Feature Selection
Removes any remaining exclude/leakage/raw-timestamp columns, verifies train/test consistency, checks for constant features and missing/infinite values, and saves the final modeling-ready datasets.

In [14]:
TRAIN_INPUT = "../data/train_numerical_features.csv"
TEST_INPUT = "../data/test_numerical_features.csv"
TRAIN_OUTPUT = "../data/train_final.csv"
TEST_OUTPUT = "../data/test_final.csv"
REPORT_OUTPUT = "../outputs/reports/final_feature_selection.csv"
TARGET = "Severity"

EXCLUDE_FEATURES = ["ID", "Country", "Turning_Loop"]
LEAKAGE_FEATURES = ["End_Lat", "End_Lng", "End_Time", "Description"]
RAW_TIMESTAMP_FEATURES = ["Start_Time", "End_Time"]

def feature_category(feature):
    if feature == TARGET: return "TARGET"
    if feature in EXCLUDE_FEATURES: return "EXCLUDE"
    if feature in LEAKAGE_FEATURES: return "LEAKAGE"
    if feature.endswith("_log1p"): return "TRANSFORMED_NUMERICAL"
    if feature.endswith("_Missing"): return "MISSING_INDICATOR"
    if feature.startswith(("Start_", "Is_", "Hour_", "Month_")): return "TEMPORAL"
    if feature.startswith(("Source_", "State_", "Timezone_", "Wind_Direction_", "Sunrise_Sunset_",
                            "Civil_Twilight_", "Nautical_Twilight_", "Astronomical_Twilight_")): return "ONE_HOT"
    if feature in ["Street", "City", "County", "Zipcode", "Airport_Code", "Weather_Condition"]: return "FREQUENCY_ENCODED"
    return "NUMERICAL"

os.makedirs("../outputs/reports", exist_ok=True)

print("\nLoading training dataset...")
train = pd.read_csv(TRAIN_INPUT)
print("Loading testing dataset...")
test = pd.read_csv(TEST_INPUT)
print(f"\nTraining shape: {train.shape}")
print(f"Testing shape : {test.shape}")

print("\nTraining target distribution:")
print(train[TARGET].value_counts().sort_index())

final_exclusions = []
for col_list in [EXCLUDE_FEATURES, LEAKAGE_FEATURES, RAW_TIMESTAMP_FEATURES]:
    for col in col_list:
        if col in train.columns:
            final_exclusions.append(col)
final_exclusions = list(dict.fromkeys(final_exclusions))
print(f"\nFinal features to remove: {final_exclusions if final_exclusions else 'None'}")

train_final = train.drop(columns=final_exclusions, errors="ignore").copy()
test_final = test.drop(columns=final_exclusions, errors="ignore").copy()

X_train = train_final.drop(columns=[TARGET])
y_train = train_final[TARGET]
X_test = test_final.drop(columns=[TARGET])
y_test = test_final[TARGET]

print(f"\nFinal predictor count: {X_train.shape[1]}")

train_only = sorted(set(X_train.columns) - set(X_test.columns))
test_only = sorted(set(X_test.columns) - set(X_train.columns))
if train_only or test_only:
    raise ValueError(f"Feature mismatch. Train only: {train_only} | Test only: {test_only}")
print("Training and testing features match.")

leakage_remaining = [c for c in X_train.columns if c in LEAKAGE_FEATURES]
if leakage_remaining:
    raise ValueError(f"Leakage features remain: {leakage_remaining}")
print("No leakage features remain.")

constant_features = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
if constant_features:
    print(f"\nConstant features found and removed: {constant_features}")
    X_train = X_train.drop(columns=constant_features)
    X_test = X_test.drop(columns=constant_features)
else:
    print("\nNo constant predictor features found.")

train_check = pd.concat([X_train, y_train], axis=1)
test_check = pd.concat([X_test, y_test], axis=1)
train_missing = int(train_check.isna().sum().sum())
test_missing = int(test_check.isna().sum().sum())
numeric_train = X_train.select_dtypes(include=[np.number])
train_inf = int(np.isinf(numeric_train.to_numpy()).sum()) if numeric_train.shape[1] > 0 else 0
print(f"\nTraining missing values: {train_missing} | infinite: {train_inf}")
print(f"Testing missing values : {test_missing}")
if train_missing > 0 or test_missing > 0:
    raise ValueError("Missing values remain in final modeling datasets.")

X_test = X_test[X_train.columns]
train_final = pd.concat([X_train, y_train], axis=1)
test_final = pd.concat([X_test, y_test], axis=1)

report_rows = [{
    "Feature": f, "Category": feature_category(f), "Data_Type": str(X_train[f].dtype),
    "Train_Unique_Values": int(X_train[f].nunique(dropna=False)), "Test_Unique_Values": int(X_test[f].nunique(dropna=False)),
} for f in X_train.columns]
report = pd.DataFrame(report_rows)
report.to_csv(REPORT_OUTPUT, index=False)

print("\nFeature category counts:")
print(report["Category"].value_counts().sort_index())

train_final.to_csv(TRAIN_OUTPUT, index=False)
test_final.to_csv(TEST_OUTPUT, index=False)

print(f"\nFinal training shape: {train_final.shape}")
print(f"Final testing shape : {test_final.shape}")
print("\nDatasets are ready for model training.")


Loading training dataset...
Loading testing dataset...

Training shape: (799673, 141)
Testing shape : (199919, 141)

Training target distribution:
Severity
1      6890
2    636908
3    134945
4     20930
Name: count, dtype: int64

Final features to remove: ['ID', 'Turning_Loop']

Final predictor count: 138
Training and testing features match.
No leakage features remain.

No constant predictor features found.

Training missing values: 0 | infinite: 0
Testing missing values : 0

Feature category counts:
Category
FREQUENCY_ENCODED         6
MISSING_INDICATOR         9
NUMERICAL                20
ONE_HOT                  88
TEMPORAL                 13
TRANSFORMED_NUMERICAL     2
Name: count, dtype: int64

Final training shape: (799673, 139)
Final testing shape : (199919, 139)

Datasets are ready for model training.
